In [ ]:
"""
Hourly Data Assimilation & Spatial Interpolation using Ordinary + Universal Kriging
-----------------------------------------------------------------------------------

This script constructs hourly, 1-km gridded predictor fields (temperature, RH, IMERG PLP,
and MRoS precipitation-phase proxy) over the DEM domain. The workflow combines:

• Ordinary Kriging (OK) applied to *temperature residuals*, after removing an
  hour-specific lapse-rate trend estimated from station data.
• Universal Kriging (UK) with DEM elevation as an external drift for *RH, IMERG PLP,
  and MRoS proxy*, allowing the interpolation to explicitly account for terrain-driven
  large-scale gradients.

Key methodological improvements:
  1. Dynamic lapse-rate estimation per hour for temperature fields, followed by OK
     on residuals only; the DEM-based trend is re-applied to reconstruct the final field.
  2. Elevation-driven drift (UK) for RH, IMERG PLP, and MRoS proxy, allowing the 
     model to represent terrain-induced gradients explicitly rather than absorbing them 
     into the variogram.
  3. Pre-calibrated variogram parameters (no hourly auto-fit), improving numerical 
     stability, consistency across hours, and computational efficiency.
  4. Vectorized kriging's backend used, enabling each hourly OK/UK system to be computed once 
     and applied across the full grid without chunked looping.
  5. DEM-integrated preprocessing, including sampling station elevations and using 
     gridded DEM elevation for both temperature re-trending and UK drift extraction.
  6. Improved handling of the MRoS proxy, including light numeric and spatial jitter 
     to preserve semivariance and avoid singular covariance matrices.

Pipeline Summary
----------------
1. CONFIG  
   Defines variables, hourly time window, kriging model settings, min-point thresholds,  
   DEM path, and fixed spherical variogram parameters.

2. UTILITIES  
   Time indexing, projection management, DEM loading/reprojection, and AOI filtering.

3. DATA INGEST  
   Loads pre-processed hourly station, IMERG, and MRoS parquet files; filters all points  
   to the DEM AOI.

4. DEM UTILITIES  
   Assigns DEM elevation to any station missing an elevation; builds 1-km grid centers  
   and flattened coordinate arrays.

5. INTERPOLATION  
   • Temperature (temp_air, temp_dew, temp_wet):
       - Estimate hourly lapse rate; compute residuals; apply Ordinary Kriging to residuals;
         reconstruct full field using DEM-based trend.
   • Non-temperature variables (RH, PLP, MRoS proxy):
       - Universal Kriging using DEM elevation as an external drift.
       - MRoS proxy undergoes controlled jitter to allow nonzero semivariance.

6. HOURLY LOOP  
   Iterates over every hour and variable, performs kriging, and populates an xarray dataset.

7. OUTPUTS  
   Produces a CF-compliant NetCDF file of shape [time * y * x] and optional quicklook maps
   showing interpolated fields with station/MRoS overlays.

Notes
-----
• Earlier versions (e.g., v4_test) used per-hour automatic variogram fitting; this version
  (“v5_test” and full production) uses fixed, calibrated variogram parameters for all
  OK and UK operations.
• Universal Kriging with elevation drift replaces earlier lapse-only detrending for
  non-temperature variables.
• The resulting predictor stack provides physically consistent fields suitable for
  downstream ML model training and precipitation-phase analysis.

"""


# ============================ IMPORTS ============================
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio as rio
import rioxarray
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
import xarray as xr
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LinearRegression # for lapse rate estimation


# Kriging
from pykrige.uk import UniversalKriging # used on IMERG, MRoS proxy, and RH
from pykrige.ok import OrdinaryKriging # used on temperature variables with dynamic lapse rate



In [42]:
BASE_DIR = Path().resolve().parent
print("BASE_DIR:", BASE_DIR)

CONFIG = {
    # Time windows
    "wy_start":  "2024-10-01T00:00:00Z",
    "wy_end":    "2025-05-31T23:59:59Z",
    "test_start": "2025-03-30T00:00:00Z",   # narrow test window first
    "test_end":   "2025-04-02T23:00:00Z",
    # "test_start": "2024-10-01T00:00:00Z",   # Entire window
    # "test_end":   "2025-05-31T23:59:59Z",

    # Paths
    "dem_path":  BASE_DIR / "DEM_1km_clipped.tif",   # final clipped DEM
    "out_dir":   BASE_DIR / "outputs/hourly_pipeline",

    # Projection fallback if DEM CRS is geographic
    "proj_fallback": "EPSG:26911",  # UTM 11N

    # Data inputs (hourly parquets produced upstream)
    "stations_parquet": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "imerg_parquet":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/imerg_hourly.parquet",
    "mros_parquet":     BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",

    # Variables
    "variables": [
        ("temp_air",       "station"),
        ("temp_dew",       "station"),
        ("temp_wet",       "station"),
        ("rh",             "station"),
        ("mros_plp_proxy", "mros"),
        ("plp",            "imerg"),
    ],

    "min_points": {  # per-variable minimum points
        "temp_air": 4, 
        "temp_dew": 4, 
        "temp_wet": 4, 
        "rh": 4,
        "mros_plp_proxy": 2, 
        "plp": 1
    },

    # Kriging/variogram
    "variogram_model": "spherical",        # keep spherical as default
  
    # format for PyKrige: [sill, range, nugget]
    # These fixed parameters were fine tuned by first running with auto-fit and determining median values, then incorporating buffer.
    "variogram_params_fixed": {
        # OK
        "temp_air_resid":  [9.44,     9_990.0,   4.72],
        "temp_dew_resid":  [6.05,     15_100.0,  1.79],
        "temp_wet_resid":  [8.71,     6_780.0,   4.36],
        # UK (w DEM drift)
        "rh":              [107.0,    148_000.0, 53.5],
        "mros_plp_proxy":  [0.161,    56_300.0,  0.00016],
        "plp":             [2090.0,   150_000.0, 2.09],
    }

}

OUT_DIR = Path(CONFIG["out_dir"]); OUT_DIR.mkdir(parents=True, exist_ok=True)
out_nc = OUT_DIR / "hourly_predictors_1km_kriging_v5_test.nc" # change name if running test or on full window

VARIOGRAM_LOG = {}
for name, _ in CONFIG["variables"]:
    VARIOGRAM_LOG[name] = []
    if name.startswith("temp_"):
        VARIOGRAM_LOG[f"{name}_resid"] = []


BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [43]:
# ============================ UTILITIES ============================
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [44]:
# ======================= Lapse-rate utilities =======================

def estimate_lapse_rate(
    st_df: pd.DataFrame,
    temp_col: str = "temp_air",
    elev_col: str = "elev",
    default_lapse: float = -0.005,
    min_points: int = 5,
    bounds: tuple = (-0.009, 0.002) # °C per meter
) -> float:
    """
    Dynamically estimate lapse rate (°C per meter) from station data.
    Returns slope (K/m).

    - default_lapse   = fallback slope (°C/m)
    - bounds          = allowable slope range (°C/m)
    """
    use = st_df.dropna(subset=[temp_col, elev_col])
    if len(use) < min_points:
        return default_lapse

    X = use[[elev_col]].values.astype(float)
    y = use[temp_col].values.astype(float)

    try:
        model = LinearRegression().fit(X, y)
        slope = float(model.coef_[0])
        # Keep slope only if within plausible limits
        if bounds[0] <= slope <= bounds[1]:
            return slope
        else:
            return default_lapse
    except Exception:
        return default_lapse


In [45]:
# --------------------- DEM Loading & Grid Setup ------------------------

def load_dem(path, target_crs):
    """
    Load DEM, reproject if needed, and return:
        dem (2D array), profile, CRS object
    Ensures DEM is ALWAYS in target_crs for consistency.
    """
    with rio.open(path) as src:
        src_crs = CRS.from_user_input(src.crs)
        tgt_crs = CRS.from_user_input(target_crs)

        # If CRS matches target CRS, no reprojection needed
        if src_crs == tgt_crs:
            dem = src.read(1).astype(np.float32)
            profile = src.profile
            profile["crs"] = tgt_crs.to_wkt()
            return dem, profile, tgt_crs

        # Otherwise reproject
        print(f"Reprojecting DEM from {src_crs} → {tgt_crs}")

        transform, width, height = calculate_default_transform(
            src_crs, tgt_crs, src.width, src.height, *src.bounds
        )

        profile = src.profile.copy()
        profile.update({
            "crs": tgt_crs.to_wkt(),
            "transform": transform,
            "width": width,
            "height": height,
        })

        dem = np.zeros((height, width), dtype=np.float32)

        reproject(
            rio.band(src, 1), dem,
            src_transform=src.transform, src_crs=src_crs,
            dst_transform=transform, dst_crs=tgt_crs,
            resampling=Resampling.bilinear,
        )

        return dem, profile, tgt_crs


# ---- Load DEM with consistent CRS ----

dem_data, dem_profile, proj_crs = load_dem(
    CONFIG["dem_path"],
    CONFIG["proj_fallback"]
)

H, W = dem_profile["height"], dem_profile["width"]
T = dem_profile["transform"]

# ---- Build coordinate centers using the affine transform ----

x_centers = T.c + (np.arange(W) + 0.5) * T.a
y_centers = T.f + (np.arange(H) + 0.5) * T.e   # T.e is usually negative for north-up

# Full grid coordinate pairs (H*W x 2)
Xg, Yg = np.meshgrid(x_centers, y_centers)     # Xg, Yg are H x W
grid_xy = np.column_stack([Xg.ravel(), Yg.ravel()])  # (H*W, 2)

# ---- Flatten DEM and identify valid cells ----

grid_elev = dem_data.ravel().astype(np.float32)
valid_points = np.isfinite(grid_elev)

grid_xy_valid = grid_xy[valid_points]
grid_elev_valid = grid_elev[valid_points]

print(f"DEM CRS: {proj_crs}")
print(f"DEM size: {W} x {H}, pixel ~{abs(T.a):.1f} m")
print(f"Valid DEM cells: {len(grid_xy_valid)}")


# -------- Create AOI bounding polygon (in WGS84) --------

def load_dem_aoi(dem_profile):
    """Return AOI bounding box in WGS84 based on the DEM profile."""
    bounds = rio.coords.BoundingBox(*rio.transform.array_bounds(
        dem_profile["height"], dem_profile["width"], dem_profile["transform"]
    ))

    aoi_wgs84 = transform_bounds(
        CRS.from_wkt(dem_profile["crs"]),
        "EPSG:4326",
        bounds.left, bounds.bottom, bounds.right, bounds.top,
        densify_pts=21
    )

    return box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])

aoi_poly = load_dem_aoi(dem_profile)


DEM CRS: EPSG:26911
DEM size: 142 x 251, pixel ~1000.0 m
Valid DEM cells: 35642


In [46]:
print("DEM stats:")
print("  NaNs:", np.isnan(dem_data).sum())
print("  +Inf:", np.isinf(dem_data).sum())
print("  -Inf:", np.isneginf(dem_data).sum())
print("  shape:", dem_data.shape)
print("  min/max:", np.nanmin(dem_data), np.nanmax(dem_data))

DEM stats:
  NaNs: 0
  +Inf: 0
  -Inf: 0
  shape: (251, 142)
  min/max: 0.0 3717.6125


In [47]:
# ============================ DATA LOADING ============================

# Load hourly parquets (already generated upstream)
st_hr   = pd.read_parquet(CONFIG["stations_parquet"])
imerg_hr = pd.read_parquet(CONFIG["imerg_parquet"])
mros_hr  = pd.read_parquet(CONFIG["mros_parquet"])

# Time to UTC and filter window
for df, time_col in [(st_hr, "hour_utc"), (imerg_hr, "hour_utc"), (mros_hr, "hour_utc")]:
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce").dt.floor("h")

HOURS = hourly_index(CONFIG["test_start"], CONFIG["test_end"])  # inclusive hourly range

# Filter to AOI bbox in lon/lat

def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr    = filter_points_to_aoi(st_hr, aoi_poly)
imerg_hr = filter_points_to_aoi(imerg_hr, aoi_poly)
mros_hr  = filter_points_to_aoi(mros_hr, aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros_hr))


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\2193629452.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),


333758 2136240 7465


In [48]:
# ============================= 4) DEM utils =============================

def add_dem_elev_if_missing(st_df: pd.DataFrame, profile, proj_crs) -> pd.DataFrame:
    """Fill missing station elevations by nearest-neighbor sampling of DEM."""
    if "elev" not in st_df.columns:
        st_df = st_df.copy(); st_df["elev"] = np.nan
    need = st_df["elev"].isna()
    if not need.any():
        return st_df

    tf = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
    xx, yy = tf.transform(st_df.loc[need, "lon"].values, st_df.loc[need, "lat"].values)
    rr, cc = rio_rowcol(profile["transform"], xx, yy, op=round)
    rr = np.clip(rr, 0, profile["height"] - 1)
    cc = np.clip(cc, 0, profile["width"]  - 1)
    st_df = st_df.copy(); st_df.loc[need, "elev"] = dem_data[rr, cc]
    return st_df

In [49]:
# ============================= 5) INTERPOLATION =============================

def _project_lonlat_to_xy(lon, lat, dst_crs):
    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    return tf.transform(lon, lat)


def _select_variogram_params(var_name: str):
    return CONFIG["variogram_params_fixed"].get(var_name, None)


def krige_residuals(hour_points: pd.DataFrame,
                    grid_xy: np.ndarray,
                    proj_crs,
                    value_col: str,
                    min_points: int) -> np.ndarray:
    """
    Ordinary Kriging of residuals (no drift terms).
    Used for temperature residuals after detrending lapse rate.
    """
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        print(f"Not enough points for residual kriging ({value_col})")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # Project lon/lat
    px, py = _project_lonlat_to_xy(pts["lon"].values, pts["lat"].values, proj_crs)

    # Determine DEM extent
    xmin, xmax = x_centers.min(), x_centers.max()
    ymin, ymax = y_centers.min(), y_centers.max()

    # Mask points that fall inside DEM grid
    mask = (px >= xmin) & (px <= xmax) & (py >= ymin) & (py <= ymax)

    pts = pts.loc[mask].copy()
    px = px[mask]
    py = py[mask]

    if len(px) == 0:
        print(f"No residual points inside DEM for {value_col}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    vals = pts[value_col].values.astype(float)

    print(f"Fitting OK for residuals of {value_col} with {len(vals)} obs...")

    try:
        OK = OrdinaryKriging(
            px, py, vals,
            variogram_model=CONFIG["variogram_model"],
            # variogram_parameters=None, # enable to AUTO-FIT for calibration of variogram parameters first
            variogram_parameters=CONFIG["variogram_params_fixed"][value_col], # now use calibrated/optimized parameters
            enable_plotting=False,
            verbose=False,
        )

        # Log fitted parameters
        params = OK.variogram_model_parameters  # [sill, range, nugget] for spherical
        print(f"  Fitted variogram params for {value_col}: {params}")
        if value_col in VARIOGRAM_LOG:
            VARIOGRAM_LOG[value_col].append(params)

    except Exception as e:
        print(f"OK init failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    try:
        grid_x = grid_xy[:, 0]
        grid_y = grid_xy[:, 1]
        z_pred, _ = OK.execute(
            "points", grid_x, grid_y,
            backend="vectorized"
        )
        return np.asarray(z_pred, dtype=np.float32)
    except Exception as e:
        print(f"Residual prediction failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

def krige_with_dem_drift(hour_points: pd.DataFrame,
                         grid_xy: np.ndarray,
                         proj_crs,
                         value_col: str,
                         min_points: int,
                         dem_data: np.ndarray,
                         x_centers: np.ndarray,
                         y_centers: np.ndarray) -> np.ndarray:
    """
    Universal Kriging with DEM elevation as external Z drift.
    ---------------------------------------------------------
    - Uses 'external_Z' drift (recommended for physical terrain effects).
    - Drift at observations = DEM elevation at station coordinates.
    - Drift at predictions = DEM elevation grid (external_drift).
    - PyKrige automatically extracts DEM drift at prediction points.

    Inputs:
        hour_points : DataFrame with columns: lon, lat, value_col
        grid_xy     : N x 2 array of grid coords (unused directly except for shapes)
        proj_crs    : CRS of the DEM
        value_col   : variable to interpolate
        min_points  : minimum required obs for UK
        dem_data    : 2D DEM array (H x W)
        x_centers   : vector of DEM x-coordinates (W,)
        y_centers   : vector of DEM y-coordinates (H,)
    """

    # 1. Filter valid points
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        print(f"Not enough points for {value_col}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # 2. Project station lon/lat → DEM CRS
    px, py = _project_lonlat_to_xy(pts["lon"].values, pts["lat"].values, proj_crs)

    # Determine DEM extent
    xmin, xmax = x_centers.min(), x_centers.max()
    ymin, ymax = y_centers.min(), y_centers.max()

    # Mask points that fall inside DEM grid
    mask = (px >= xmin) & (px <= xmax) & (py >= ymin) & (py <= ymax)

    pts = pts.loc[mask].copy()
    px = px[mask]
    py = py[mask]

    if len(px) == 0:
        print(f"No residual points inside DEM for {value_col}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # 3. Sample DEM elevation at those projected station points
    rr, cc = rio_rowcol(
        dem_profile["transform"],
        px, py,  # station projected coords
        op=round
    )
    rr = np.clip(rr, 0, dem_data.shape[0] - 1)
    cc = np.clip(cc, 0, dem_data.shape[1] - 1)
    obs_elev = dem_data[rr, cc].astype(float)   # drift at observations (Z)

    vals = pts[value_col].values.astype(float)

    print(f"Fitting UK for {value_col} with {len(vals)} obs...")

    # 4. Build UK model with DEM-based external drift
    try:
        UK = UniversalKriging(
            px, py, vals,
            variogram_model=CONFIG["variogram_model"],
            # variogram_parameters=None,          # enable to AUTO-FIT for calibration of variogram parameters first
            variogram_parameters=CONFIG["variogram_params_fixed"][value_col], # now use calibrated/optimized parameters
            drift_terms=["external_Z"],         # DEM elevation drift
            external_drift=dem_data,            # full DEM grid (H x W)
            external_drift_x=x_centers,         # x-coordinates W
            external_drift_y=y_centers          # y-coordinates H
        )
        
        # log fitted params
        params = UK.variogram_model_parameters  # PyKrige exposes this
        print(f"  Fitted variogram params for {value_col}: {params}")
        VARIOGRAM_LOG[value_col].append(params)

    except Exception as e:
        print(f"UK init failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)

    # 5. Predict entire grid (DEM drift extracted internally)
    try:
        grid_x = grid_xy[:, 0]
        grid_y = grid_xy[:, 1]

        z_pred, _ = UK.execute(
            "points", grid_x, grid_y,
            backend="vectorized"   # optional, fast
        )
        # builds the entire kriging system (covariance matrix) and predicts all 35k points in one go.
        # This means all pairwise distances between 40 obs → 35,278 grid points are in memory simultaneously.
        
        return np.asarray(z_pred, dtype=np.float32)

    except Exception as e:
        print(f"Prediction failed for {value_col}: {e}")
        return np.full(grid_xy.shape[0], np.nan, dtype=np.float32)


In [50]:
# ============================= 6) HOURLY LOOP =============================

coords = {"time": HOURS, "y": y_centers, "x": x_centers}
var_names = [v[0] for v in CONFIG["variables"]]
data_vars = {name: np.full((len(HOURS), H, W), np.nan, dtype=np.float32) for name in var_names}

for ti, t in enumerate(tqdm(HOURS, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]
    mros_t = mros_hr[mros_hr["hour_utc"] == t]

    print(f"\n[{t}] counts:")
    print("  stations:", len(st_t),
          " temp_air:", st_t["temp_air"].notna().sum() if "temp_air" in st_t else 0,
          " rh:",      st_t["rh"].notna().sum()        if "rh"      in st_t else 0)
    print("  IMERG:", len(imerg_t),
          " plp:", imerg_t["plp"].notna().sum() if "plp" in imerg_t else 0)
    print("  MRoS:", len(mros_t),
          " proxy:", mros_t["mros_plp_proxy"].notna().sum() if "mros_plp_proxy" in mros_t else 0)

    # Ensure station elevs present for lapse
    st_t = add_dem_elev_if_missing(st_t, dem_profile, proj_crs)

    for name, src in CONFIG["variables"]:
        min_pts = CONFIG["min_points"].get(name, 3)

        if src == "station":
            if name not in st_t.columns:
                continue
            pts = st_t[["lon", "lat", "elev", name]].dropna(subset=[name])
        elif src == "imerg":
            pts = imerg_t.rename(columns={"plp": name})[["lon", "lat", name]].assign(elev=0.0)
        elif src == "mros":
            pts = mros_t.rename(columns={"mros_plp_proxy": name})[["lon", "lat", name]].assign(elev=0.0)
        else:
            continue

        if pts[name].notna().sum() < min_pts:
            print(f"    {name}: insufficient points ({pts[name].notna().sum()} < {min_pts})")
            continue
        
        # HANDLING FOR MRoS PROXY (discrete / constant cases): fractionalize and introduce small random noise to preserve ordinal meaning but allow nonzero semivariance
        if name == "mros_plp_proxy":
            # add jitter
            pts[name] = pts[name]/100.0 + np.random.uniform(-0.03, 0.03, len(pts))

            # coordinate jitter: +/- 200 m (tune as needed)
            jitter_xy = 200.0
            dx = np.random.uniform(-jitter_xy, jitter_xy, len(pts))
            dy = np.random.uniform(-jitter_xy, jitter_xy, len(pts))

            # apply in projected space
            tf = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
            px, py = tf.transform(pts["lon"].values, pts["lat"].values)
            px_jit, py_jit = px + dx, py + dy
            lon_jit, lat_jit = Transformer.from_crs(
                proj_crs, "EPSG:4326", always_xy=True
            ).transform(px_jit, py_jit)

            pts["lon"] = lon_jit
            pts["lat"] = lat_jit

            print(f"MRoS proxy variance @ {t}: {pts[name].var():.4f}")

        try:
            # ================= TEMP VARIABLES: lapse-rate detrend + residual OK =================
            if name in ("temp_air", "temp_dew", "temp_wet"):

                # 1) Estimate HOURLY lapse rate from stations
                b_lr = estimate_lapse_rate(
                    pts,
                    temp_col=name,
                    elev_col="elev",
                    default_lapse=-0.005,        # -5 K/km fallback
                    min_points=5,
                    bounds=(-0.009, 0.002)       # physically plausible
                )
                # Intercept a computed from mean(T) - b*mean(z)
                a_lr = float(pts[name].mean() - b_lr * pts["elev"].mean())

                print(f"    {name}: using lapse rate {b_lr*1000:.2f} K/km (intercept {a_lr:.2f})")

                # 2) Compute station residuals
                pts = pts.copy()
                resid_col = f"{name}_resid"
                pts[resid_col] = pts[name] - (a_lr + b_lr * pts["elev"])

                # 3) Ordinary kriging on residuals
                resid_grid = krige_residuals(
                    hour_points=pts,
                    grid_xy=grid_xy_valid,
                    proj_crs=proj_crs,
                    value_col=resid_col,
                    min_points=min_pts,
                )

                # 4) Add lapse-plane back onto DEM grid
                trend_grid = a_lr + b_lr * grid_elev_valid
                vals = trend_grid + resid_grid

            # ================= NON-TEMP VARIABLES: UK + DEM drift =================
            else:
                vals = krige_with_dem_drift(
                    hour_points=pts,
                    grid_xy=grid_xy_valid,
                    proj_crs=proj_crs,
                    value_col=name,
                    min_points=min_pts,
                    dem_data=dem_data,
                    x_centers=x_centers,
                    y_centers=y_centers,
                )

                # Special handling for MRoS proxy (put back on 0–100 scale)
                if name == "mros_plp_proxy":
                    vals = np.clip(vals * 100.0, 0.0, 100.0)

            if len(pts):
                print(
                    f"    {name} obs stats: "
                    f"min={pts[name].min():.3f}, max={pts[name].max():.3f}, "
                    f"mean={pts[name].mean():.3f}, var={pts[name].var():.6f}"
                )

        except Exception as e:
            print(f"Kriging failed for {name} @ {t}: {e}")
            vals = np.full(grid_elev_valid.shape, np.nan, dtype=np.float32)

        
        vals_full = np.full(H * W, np.nan, dtype=np.float32)
        vals_full[valid_points] = vals
        data_vars[name][ti, :, :] = vals_full.reshape(H, W)


Hourly surfaces:   0%|                                           | 0/96 [00:00<?, ?it/s]


[2025-03-30 00:00:00+00:00] counts:
  stations: 41  temp_air: 41  rh: 41
  IMERG: 0  plp: 0
  MRoS: 0  proxy: 0
    temp_air: using lapse rate -3.28 K/km (intercept 12.25)
Fitting OK for residuals of temp_air_resid with 40 obs...
  Fitted variogram params for temp_air_resid: [4.72, 9990.0, 4.72]
    temp_air obs stats: min=-1.278, max=17.389, mean=5.079, var=22.027174
    temp_dew: using lapse rate -5.42 K/km (intercept 4.11)
Fitting OK for residuals of temp_dew_resid with 40 obs...
  Fitted variogram params for temp_dew_resid: [4.26, 15100.0, 1.79]
    temp_dew obs stats: min=-17.222, max=1.852, mean=-7.762, var=21.418239
    temp_wet: using lapse rate -2.99 K/km (intercept 6.32)
Fitting OK for residuals of temp_wet_resid with 40 obs...
  Fitted variogram params for temp_wet_resid: [4.3500000000000005, 6780.0, 4.36]
    temp_wet obs stats: min=-7.320, max=8.702, mean=-0.229, var=13.479050
Fitting UK for rh with 40 obs...
  Fitted variogram params for rh: [53.5, 148000.0, 53.5]


Hourly surfaces:   1%|▎                                  | 1/96 [00:02<03:37,  2.29s/it]

    rh obs stats: min=15.000, max=79.000, mean=36.410, var=194.206877
    mros_plp_proxy: insufficient points (0 < 2)
    plp: insufficient points (0 < 1)

[2025-03-30 01:00:00+00:00] counts:
  stations: 41  temp_air: 41  rh: 41
  IMERG: 0  plp: 0
  MRoS: 0  proxy: 0
    temp_air: using lapse rate -3.46 K/km (intercept 11.54)
Fitting OK for residuals of temp_air_resid with 40 obs...
  Fitted variogram params for temp_air_resid: [4.72, 9990.0, 4.72]
    temp_air obs stats: min=-2.389, max=16.722, mean=3.957, var=22.307709
    temp_dew: using lapse rate -6.07 K/km (intercept 5.84)
Fitting OK for residuals of temp_dew_resid with 40 obs...
  Fitted variogram params for temp_dew_resid: [4.26, 15100.0, 1.79]
    temp_dew obs stats: min=-16.667, max=2.222, mean=-7.441, var=23.799653
    temp_wet: using lapse rate -3.21 K/km (intercept 6.31)
Fitting OK for residuals of temp_wet_resid with 40 obs...
  Fitted variogram params for temp_wet_resid: [4.3500000000000005, 6780.0, 4.36]
    temp_wet ob

Hourly surfaces:   2%|▋                                  | 2/96 [00:04<03:37,  2.31s/it]

...

    plp obs stats: min=0.000, max=100.000, mean=40.971, var=1923.844194

[2025-04-02 22:00:00+00:00] counts:
  stations: 40  temp_air: 40  rh: 40
  IMERG: 414  plp: 414
  MRoS: 6  proxy: 6
    temp_air: using lapse rate -4.64 K/km (intercept 10.69)
Fitting OK for residuals of temp_air_resid with 39 obs...
  Fitted variogram params for temp_air_resid: [4.72, 9990.0, 4.72]
    temp_air obs stats: min=-8.111, max=16.389, mean=0.468, var=29.523755
    temp_dew: using lapse rate -5.40 K/km (intercept 4.40)
Fitting OK for residuals of temp_dew_resid with 39 obs...
  Fitted variogram params for temp_dew_resid: [4.26, 15100.0, 1.79]
    temp_dew obs stats: min=-13.520, max=2.778, mean=-7.503, var=16.269394
    temp_wet: using lapse rate -3.48 K/km (intercept 4.57)
Fitting OK for residuals of temp_wet_resid with 39 obs...
  Fitted variogram params for temp_wet_resid: [4.3500000000000005, 6780.0, 4.36]
    temp_wet obs stats: min=-14.091, max=8.620, mean=-3.107, var=23.306659
Fitting UK for rh w

Hourly surfaces:  99%|█████████████████████████████████▋| 95/96 [12:52<00:15, 15.19s/it]

    plp obs stats: min=0.000, max=100.000, mean=40.971, var=1923.844194

[2025-04-02 23:00:00+00:00] counts:
  stations: 41  temp_air: 41  rh: 41
  IMERG: 414  plp: 414
  MRoS: 1  proxy: 1
    temp_air: using lapse rate -4.55 K/km (intercept 10.25)
Fitting OK for residuals of temp_air_resid with 40 obs...
  Fitted variogram params for temp_air_resid: [4.72, 9990.0, 4.72]
    temp_air obs stats: min=-7.611, max=15.778, mean=0.299, var=27.642887
    temp_dew: using lapse rate -5.19 K/km (intercept 4.07)
Fitting OK for residuals of temp_dew_resid with 40 obs...
  Fitted variogram params for temp_dew_resid: [4.26, 15100.0, 1.79]
    temp_dew obs stats: min=-12.437, max=3.148, mean=-7.290, var=14.660307
    temp_wet: using lapse rate -3.65 K/km (intercept 4.75)
Fitting OK for residuals of temp_wet_resid with 40 obs...
  Fitted variogram params for temp_wet_resid: [4.3500000000000005, 6780.0, 4.36]
    temp_wet obs stats: min=-12.568, max=8.269, mean=-3.235, var=22.020053
Fitting UK for rh w

Hourly surfaces: 100%|██████████████████████████████████| 96/96 [13:04<00:00,  8.18s/it]

    plp obs stats: min=0.000, max=100.000, mean=40.971, var=1923.844194


In [51]:
# =========================== SAVE NETCDF ==============================

# ---- 1. Build Dataset ----
ds = xr.Dataset(
    {
        **{k: xr.DataArray(v, coords=coords, dims=("time","y","x"))
           for k, v in data_vars.items()},
        "elev": xr.DataArray(
            dem_data.astype(np.float32),
            coords={"y": y_centers, "x": x_centers},
            dims=("y","x"),
            attrs={"units": "m", "long_name": "DEM elevation"}
        ),
    },
    attrs={
        "title": "Hourly predictor stacks on 1-km grid (Universal Kriging w/ elevation drift)",
        "interpolation_method": "Universal Kriging (external drift = DEM elevation)",
        "variogram_model": CONFIG["variogram_model"],
        # "variogram_strategy": CONFIG["variogram_strategy"],
        "test_window": f"{CONFIG['test_start']} → {CONFIG['test_end']}",
    }
)

# ---- 2. Assign CF spatial dimensions ----
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

# ---- 3. Write CRS using EPSG, not CRS object ----
crs_obj = CRS.from_user_input(dem_profile["crs"])
epsg_code = crs_obj.to_epsg()
if epsg_code is None:
    raise ValueError(f"Could not derive EPSG from CRS: {dem_profile['crs']}")
ds = ds.rio.write_crs(epsg_code, grid_mapping_name="spatial_ref")

# ---- 4. Write GeoTransform ----
ds = ds.rio.write_transform(dem_profile["transform"])

# force consistency
A = dem_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

# ensure grid_mapping attribute exists
for v in ds.data_vars:
    ds[v].attrs["grid_mapping"] = "spatial_ref"

# ---- 5. Ensure time has no timezone ----
if hasattr(ds.indexes.get("time", None), "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# ---- 6. Encoding & Chunking ----
def _chunks_for(da):
    if da.ndim == 3 and da.dims == ("time","y","x"):
        return (min(24, da.shape[0]), min(256, da.shape[1]), min(256, da.shape[2]))
    if da.ndim == 2 and da.dims == ("y","x"):
        return (min(256, da.shape[0]), min(256, da.shape[1]))
    return None

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    enc = {"zlib": True, "complevel": 4}
    if ch:
        enc["chunksizes"] = ch
    encoding[name] = enc

# ---- 7. Write NetCDF ----
ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)

print(f"Wrote {out_nc} successfully.")
ds.close()


Wrote C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\hourly_predictors_1km_kriging_v5_test.nc successfully.


In [52]:
# -------------------- Quick Plotting ------------------------------------

from pyproj import CRS

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")   # disable 1e6 scientific format
        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # overlay
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------

quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

# day = "2025-03-04"
# all_times = pd.to_datetime(ds.time.values).floor("h")  # dataset times
# mask = all_times.normalize() == pd.to_datetime(day)
# sample_hours = all_times[mask]
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//20)]
# print(f"Found {len(sample_hours)} timesteps on {day}")

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")  # tz-naive

    # Ensure obs times are made tz-naive before comparison
    st_t   = st_hr[st_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]
    mros_t = mros_hr[mros_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t_floor, st_t, mros_t,
                   out_png=quick_dir / f"test5_univ_kriging_quick_{print_time(t_floor).replace(':','-')}.png")


[2025-03-30 00:00:00] Stations: 41, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test5_univ_kriging_quick_2025-03-30 00-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-03-30 04:00:00] Stations: 40, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test5_univ_kriging_quick_2025-03-30 04-00Z.png | plotted 39 stations, 0 MRoS (clipped to DEM)
[2025-03-30 08:00:00] Stations: 41, MRoS: 0


...

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test5_univ_kriging_quick_2025-04-02 12-00Z.png | plotted 40 stations, 1 MRoS (clipped to DEM)
[2025-04-02 16:00:00] Stations: 41, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test5_univ_kriging_quick_2025-04-02 16-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-04-02 20:00:00] Stations: 41, MRoS: 5


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_34220\1918148602.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test5_univ_kriging_quick_2025-04-02 20-00Z.png | plotted 40 stations, 5 MRoS (clipped to DEM)


In [53]:
## Run only under variogram auto-fit to identify calibrated parameters
# print("\n========== Variogram parameter summary (auto-fit) ==========")
# for var, plist in VARIOGRAM_LOG.items():
#     if not plist:
#         print(f"{var}: no fits logged.")
#         continue
#     arr = np.vstack(plist)  # shape (n_hours, 3) → [sill, range, nugget]

#     # basic stats
#     med = np.median(arr, axis=0)
#     p10 = np.percentile(arr, 10, axis=0)
#     p90 = np.percentile(arr, 90, axis=0)

#     print(
#         f"{var}: median = [sill={med[0]:.3g}, range={med[1]:.3g}, nugget={med[2]:.3g}], "
#         f"10–90% ranges: "
#         f"sill[{p10[0]:.3g}–{p90[0]:.3g}], "
#         f"range[{p10[1]:.3g}–{p90[1]:.3g}], "
#         f"nugget[{p10[2]:.3g}–{p90[2]:.3g}]"
#     )
